In [ ]:
from deepface import DeepFace
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import time
import os

In [2]:
#cattura dell'immagine live dalla camera
cap = cv2.VideoCapture(0)
db = "C:/Users/fulvi/Documents/università/magistrale/db_robot"
face_match = True
emotion = "sconosciuta"


In [ ]:
def face_match_function(target, path_db, model_name='VGG-Face'):
  global face_match
  try:
        # DeepFace.find calcola la similarità e ordina i risultati per distanza
        # Distanza minore = maggiore somiglianza
        risultati = DeepFace.find(
            img_path=target,
            db_path=path_db,
            model_name=model_name,
            enforce_detection=False,  # Non solleva errore se non trova il volto
            silent=True
        )

        # risultati è una lista di DataFrame, prendiamo il primo
        if len(risultati) > 0 and not risultati[0].empty:
            print("Match trovato! Il volto è già nel database.")
            face_match = True
        else:
            #salva l'immagine nel db
            # 1. Creiamo un nome file univoco basato sul timestamp attuale
            new_image = f"img_{int(time.time())}.jpg"
            path = os.path.join(path_db, new_image)
            
            # 2. Salviamo l'immagine
            cv2.imwrite(path, target)
            print(f"Immagine salvata in: {path}")
            
            # 3. Eliminiamo il file di cache (.pkl) di DeepFace
            # Se non lo facciamo, al prossimo avvio non vedrà la nuova immagine salvata
            cache_file = os.path.join(path_db, f"representations_{model_name.lower().replace('-', '_')}.pkl")
            if os.path.exists(cache_file):
                os.remove(cache_file)
                
        print("Emotion recognition ...")
        emotion_match = DeepFace.analyze(
            img_path=target,
            actions=['emotion'],
            enforce_detection=False,
            silent=True
        ) 
        emotion = emotion_match[0]['dominant_emotion']
        print(f"L'utente si sente: {emotion}")
  except ValueError:
        return False
  return face_match,emotion

In [ ]:
# Controlla se la webcam è aperta correttamente
if not cap.isOpened():
    print("Impossibile aprire la webcam")
    exit()
    
# 2. Cattura un fotogramma (immagine)
# 'ret' è un booleano (True/False), 'frame' è l'immagine vera e propria
ret, frame = cap.read()

# 3. Se la cattura è andata a buon fine, il frame è salvato nella variabile
if ret:
    print("Immagine catturata correttamente nella variabile 'frame'")
    # Opzionale: mostra l'immagine catturata
    cv2.imshow('Immagine Catturata', frame)
    cv2.waitKey(0) # Attendi la pressione di un tasto
else:
    print("Impossibile leggere il fotogramma")

if ret:
    # Richiamiamo la funzione passandogli il frame appena catturato e la cartella DB
    utente, emozione = face_match_function(frame, db)
    print(f"Risultato finale: {utente}")
    print(f"stato emotivo rilevato: {emozione}")

cap.release()
cv2.destroyAllWindows()

Impossibile aprire la webcam
Impossibile leggere il fotogramma


: 